In [2]:
from datasets import load_dataset


# load the ragbench dataset
ragbench = {}
for dataset in ['covidqa', 'cuad', 'expertqa', 'finqa', 'techqa']:
  ragbench[dataset] = load_dataset("rungalileo/ragbench", dataset, split="test")

In [3]:
ragbench

{'covidqa': Dataset({
     features: ['id', 'question', 'documents', 'response', 'generation_model_name', 'annotating_model_name', 'dataset_name', 'documents_sentences', 'response_sentences', 'sentence_support_information', 'unsupported_response_sentence_keys', 'adherence_score', 'overall_supported_explanation', 'relevance_explanation', 'all_relevant_sentence_keys', 'all_utilized_sentence_keys', 'trulens_groundedness', 'trulens_context_relevance', 'ragas_faithfulness', 'ragas_context_relevance', 'gpt3_adherence', 'gpt3_context_relevance', 'gpt35_utilization', 'relevance_score', 'utilization_score', 'completeness_score'],
     num_rows: 246
 }),
 'cuad': Dataset({
     features: ['id', 'question', 'documents', 'response', 'generation_model_name', 'annotating_model_name', 'dataset_name', 'documents_sentences', 'response_sentences', 'sentence_support_information', 'unsupported_response_sentence_keys', 'adherence_score', 'overall_supported_explanation', 'relevance_explanation', 'all_releva

In [10]:
# !pip install -q -U google-generativeai

In [4]:
import google.generativeai as genai
import os

api_key = os.getenv("GOOGLE_API_KEY")
if not api_key:
    raise ValueError("GOOGLE_API_KEY environment variable not set. Please set it and restart the kernel.")

genai.configure(api_key=api_key)

print("Gemini API key configured successfully.")

Gemini API key configured successfully.


In [9]:
def is_time_sensitive(question):
    """
    Classifies whether a question is time-sensitive using the Gemini API.

    Args:
        question (str): The question to classify.

    Returns:
        bool: True if the question is time-sensitive, False otherwise.
    """
    model = genai.GenerativeModel('gemini-2.5-flash')
    prompt = (f"Classify the following question as time-sensitive or not. "
              f"A time-sensitive question is one that requires specific temporal context "
              f"(like 'current', 'latest', 'recent', or a specific year/date relative to now) "
              f"to be answered correctly. "
              f"Return ONLY 'True' if it is time-sensitive, and 'False' otherwise.\n\n"
              f"Question: {question}")
    
    try:
        response = model.generate_content(prompt)
        result = response.text.strip().lower()
        return result == 'true'
    except Exception as e:
        print(f"Error processing question '{question}': {e}")
        return False

In [10]:
test_results = []

# Iterate through all examples of the expertqa dataset
print("Testing classification on all ExpertQA questions...")
for example in ragbench['expertqa']:
    question = example['question']
    is_sensitive = is_time_sensitive(question)
    
    print(f"Question: {question[:100]}...")
    print(f"Time-sensitive: {is_sensitive}\n")
    
    test_results.append({
        'question': question,
        'is_time_sensitive': is_sensitive
    })

print(f"Finished testing. Processed {len(test_results)} questions.")

Testing classification on all ExpertQA questions...
Question: What are the benefits of using myocardial perfusion imaging prior to angio?...
Time-sensitive: False

Question: It is known that climate change is source of inequality. Could the solutions to combat climate chang...
Time-sensitive: False

Question: What is the clinical use of catecholamine measurement?...
Time-sensitive: False

Question: If we'd have more supplies in and out the classromm, could we reach earlier the real meaning of educ...
Time-sensitive: False

Question: What first-line tests would you request for the management of sarcoidosis?...
Time-sensitive: False

Question: SUMMARIZE THE HISTORY OF POP ART IN AMERICAN DESIGN...
Time-sensitive: False

Question: Would it be dangerous for a human being to contact directly with imidazolium based ionic liquids in ...
Time-sensitive: False

Question: How would you promote dignity and respect in care?...
Time-sensitive: False

Question: Why does the word "no" itself not exis

In [15]:
# Extract the time-sensitive results
is_time_sensitive_column = [result['is_time_sensitive'] for result in test_results]

# Add the new column to the dataset
# Ensure the number of results matches the dataset length
if len(is_time_sensitive_column) == len(ragbench['expertqa']):
    ragbench['expertqa'] = ragbench['expertqa'].add_column('is_time_sensitive', is_time_sensitive_column)
    print("Successfully added 'is_time_sensitive' column to 'expertqa' dataset.")
else:
    print(f"Error: Mismatch in length between results ({len(is_time_sensitive_column)}) and dataset ({len(ragbench['expertqa'])}).")

# Display the updated dataset with the new column
ragbench['expertqa']

Successfully added 'is_time_sensitive' column to 'expertqa' dataset.


Dataset({
    features: ['id', 'question', 'documents', 'response', 'generation_model_name', 'annotating_model_name', 'dataset_name', 'documents_sentences', 'response_sentences', 'sentence_support_information', 'unsupported_response_sentence_keys', 'adherence_score', 'overall_supported_explanation', 'relevance_explanation', 'all_relevant_sentence_keys', 'all_utilized_sentence_keys', 'trulens_groundedness', 'trulens_context_relevance', 'ragas_faithfulness', 'ragas_context_relevance', 'gpt3_adherence', 'gpt3_context_relevance', 'gpt35_utilization', 'relevance_score', 'utilization_score', 'completeness_score', 'is_time_sensitive'],
    num_rows: 203
})

In [14]:
# Display only the time-sensitive questions
time_sensitive_questions = [
    result['question'] 
    for result in test_results 
    if result['is_time_sensitive']
]

len(time_sensitive_questions)

36

In [26]:
from tqdm.auto import tqdm
import numpy as np

# Loop through all datasets in the ragbench
for dataset_name, dataset in ragbench.items():
    print(f"Processing dataset: {dataset_name}...")
    
    # Skip if the column already exists
    if 'is_time_sensitive' in dataset.column_names:
        # If you want to re-run, you might need to remove the column first
        # ragbench[dataset_name] = dataset.remove_columns(['is_time_sensitive'])
        print(f"'{dataset_name}' already has 'is_time_sensitive' column. Skipping.")
        continue

    time_sensitive_results = []
    positive_samples_count = 0
    
    # Use tqdm for a progress bar
    for example in tqdm(dataset, desc=f"Classifying {dataset_name}"):
        if positive_samples_count >= 30:
            # Stop classifying and mark the rest as None (which datasets handles as null)
            is_sensitive = None
        else:
            question = example['question']
            is_sensitive = is_time_sensitive(question)
            if is_sensitive:
                positive_samples_count += 1
        
        time_sensitive_results.append(is_sensitive)
    
    # Add the new column to the dataset
    if len(time_sensitive_results) == len(dataset):
        ragbench[dataset_name] = dataset.add_column('is_time_sensitive', time_sensitive_results)
        print(f"Successfully added 'is_time_sensitive' column to '{dataset_name}' dataset. Found {positive_samples_count} positive samples.\n")
    else:
        print(f"Error: Mismatch in length for '{dataset_name}'. Could not add column.\n")

# Display the updated ragbench dictionary with new columns
ragbench

Processing dataset: covidqa...
'covidqa' already has 'is_time_sensitive' column. Skipping.
Processing dataset: cuad...
'cuad' already has 'is_time_sensitive' column. Skipping.
Processing dataset: expertqa...
'expertqa' already has 'is_time_sensitive' column. Skipping.
Processing dataset: finqa...


Classifying finqa: 100%|██████████| 2294/2294 [05:20<00:00,  7.16it/s] 


Successfully added 'is_time_sensitive' column to 'finqa' dataset. Found 30 positive samples.

Processing dataset: techqa...


Classifying techqa: 100%|██████████| 314/314 [10:37<00:00,  2.03s/it]


Successfully added 'is_time_sensitive' column to 'techqa' dataset. Found 30 positive samples.



{'covidqa': Dataset({
     features: ['id', 'question', 'documents', 'response', 'generation_model_name', 'annotating_model_name', 'dataset_name', 'documents_sentences', 'response_sentences', 'sentence_support_information', 'unsupported_response_sentence_keys', 'adherence_score', 'overall_supported_explanation', 'relevance_explanation', 'all_relevant_sentence_keys', 'all_utilized_sentence_keys', 'trulens_groundedness', 'trulens_context_relevance', 'ragas_faithfulness', 'ragas_context_relevance', 'gpt3_adherence', 'gpt3_context_relevance', 'gpt35_utilization', 'relevance_score', 'utilization_score', 'completeness_score', 'is_time_sensitive'],
     num_rows: 246
 }),
 'cuad': Dataset({
     features: ['id', 'question', 'documents', 'response', 'generation_model_name', 'annotating_model_name', 'dataset_name', 'documents_sentences', 'response_sentences', 'sentence_support_information', 'unsupported_response_sentence_keys', 'adherence_score', 'overall_supported_explanation', 'relevance_expl

In [ ]:
import json
import random
import os

big_corpus = []

# Loop through all datasets in the ragbench
for dataset_name, dataset in ragbench.items():
    print(f"Processing {dataset_name}...")

    # Filter for time-sensitive and non-time-sensitive questions
    time_sensitive_ds = dataset.filter(lambda x: x['is_time_sensitive'] is True)
    non_time_sensitive_ds = dataset.filter(lambda x: x['is_time_sensitive'] is False)

    # Get the first 30 time-sensitive samples
    time_sensitive_samples = time_sensitive_ds.select(range(min(30, len(time_sensitive_ds))))
    
    # Randomly select 30 non-time-sensitive samples
    if len(non_time_sensitive_ds) >= 30:
        non_time_sensitive_samples = non_time_sensitive_ds.shuffle(seed=42).select(range(30))
    else:
        non_time_sensitive_samples = non_time_sensitive_ds
    
    print(f"  - Found {len(time_sensitive_samples)} time-sensitive samples.")
    print(f"  - Selected {len(non_time_sensitive_samples)} non-time-sensitive samples.")

    # Assign new IDs and add to the big corpus
    id_counter = 1
    for sample in time_sensitive_samples:
        sample['id'] = f"{dataset_name}_{id_counter:02d}"
        sample['source_dataset'] = dataset_name
        sample['is_fresh'] = True
        big_corpus.append(sample)
        id_counter += 1
        
    for sample in non_time_sensitive_samples:
        sample['id'] = f"{dataset_name}_{id_counter:02d}"
        sample['source_dataset'] = dataset_name
        sample['is_fresh'] = True
        big_corpus.append(sample)
        id_counter += 1

# Save the combined corpus to a JSONL file
os.makedirs("../data", exist_ok=True)
output_filename = "../data/time_sensitivity_corpus.jsonl"
with open(output_filename, 'w') as f:
    for entry in big_corpus:
        f.write(json.dumps(entry) + '\n')

print(f"\nCorpus saved to {output_filename}. Total pairs: {len(big_corpus)}")

## query json

In [48]:
def get_answer_bearing_docs(relevant_sentence_keys):
    """Convert sentence keys like ['0d', '0e', '2b'] 
    to answer-bearing doc indices {0, 2}"""
    doc_indices = set()
    for key in relevant_sentence_keys:
        doc_index = int(key[0])  # first character is doc index
        doc_indices.add(doc_index)
    return doc_indices

In [ ]:
import json

# Load the big corpus
corpus_filename = "../data/time_sensitivity_corpus.jsonl"
with open(corpus_filename, 'r') as f:
    big_corpus = [json.loads(line) for line in f]

queries = []
for item in big_corpus:
    query_id = item.get('id')
    
    # Transform sentence keys to answer-bearing doc IDs
    answer_bearing_doc_ids = []
    relevant_sentence_keys = item.get("all_relevant_sentence_keys")
    if relevant_sentence_keys:
        doc_indices = get_answer_bearing_docs(relevant_sentence_keys)
        answer_bearing_doc_ids = [f"{query_id}_doc_{i}" for i in sorted(list(doc_indices))]

    query_data = {
        "query_id" : query_id,
        "question": item.get("question"),
        "ground_truth": item.get("response"),
        "domain": item.get("source_dataset"), # Using source_dataset as domain
        "time_sensitive": item.get("is_time_sensitive"),
        "answer_bearing_doc_ids": answer_bearing_doc_ids, 
        "ragbench_adherence_score": item.get("adherence_score"),
        "ragbench_relevance_score": item.get("relevance_score")
        # The other fields from the example are not in the source data.
    }
    queries.append(query_data)

# Save the new queries to queries.jsonl
output_filename = "../data/queries.jsonl"
with open(output_filename, 'w') as f:
    for query in queries:
        f.write(json.dumps(query) + '\n')

print(f"Extracted {len(queries)} queries and saved to {output_filename}")

## corpus json

In [ ]:
import json

# Load the big corpus again
corpus_filename = "../data/time_sensitivity_corpus.jsonl"
with open(corpus_filename, 'r') as f:
    big_corpus = [json.loads(line) for line in f]

unfolded_corpus = []
for item in big_corpus:
    query_id = item.get("id")
    relevant_sentence_keys = item.get("all_relevant_sentence_keys")
    
    # Get the set of answer-bearing document indices
    if relevant_sentence_keys:
        answer_bearing_doc_indices = get_answer_bearing_docs(relevant_sentence_keys)
    else:
        answer_bearing_doc_indices = set()

    # The 'documents' field is a list of strings.
    documents = item.get("documents", [])
    if not documents:
        continue

    # Unfold documents and add 'is_answer_bearing' flag
    for i, doc_text in enumerate(documents):
        is_bearing = i in answer_bearing_doc_indices
        
        # Generate a document ID
        doc_id = f"{query_id}_doc_{i}"

        unfolded_doc = {
            "query_id": query_id,
            "doc_id": doc_id,
            "text": doc_text, # The whole string is the text
            "is_answer_bearing": is_bearing,
            "time_sensitive": item.get("is_time_sensitive"),
            "is_fresh": item.get("is_fresh") # Add time sensitivity flag to each document
        }
        unfolded_corpus.append(unfolded_doc)

# Save the new unfolded corpus to corpus.jsonl
output_filename = "../data/corpus.jsonl"
with open(output_filename, 'w') as f:
    for doc_entry in unfolded_corpus:
        f.write(json.dumps(doc_entry) + '\n')

print(f"Created unfolded corpus with {len(unfolded_corpus)} documents and saved to {output_filename}")